In [ ]:
import re
import time
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException as SeleniumTimeout
from webdriver_manager.chrome import ChromeDriverManager


def obter_segundo_controle_acoes_usuario(driver, timeout_s):
    """#acoes_usuario: 2º botão/link (o site mistura <button> e <a>; às vezes só 1 <button>)."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            # Caminho original que você gravou (filhos diretos)
            els = driver.find_elements(
                By.XPATH, '//*[@id="acoes_usuario"]/button[2]'
            )
            if els:
                return els[0]
            els = driver.find_elements(
                By.XPATH, '(//*[@id="acoes_usuario"]//button)[2]'
            )
            if els:
                return els[0]
            c = driver.find_element(By.ID, "acoes_usuario")
            diretos = c.find_elements(By.XPATH, "./button")
            if len(diretos) >= 2:
                return diretos[1]
            todos_btn = c.find_elements(By.XPATH, ".//button")
            if len(todos_btn) >= 2:
                return todos_btn[1]
            misto = c.find_elements(By.XPATH, ".//*[self::button or self::a]")
            vis = [e for e in misto if e.is_displayed()]
            if len(vis) >= 2:
                return vis[1]
        except Exception:
            pass
        time.sleep(0.35)
    raise SeleniumTimeout(
        "Não encontrou o 2º controle em #acoes_usuario após "
        f"{timeout_s}s (nota pode ter só uma ação ou layout diferente)."
    )


URL_CONSULTA_NOTAS = "https://maringa.fintel.com.br/ConsultaNotasFiscaisEmitidas"


def ir_para_consulta_notas(driver, timeout=30):
    """Volta à lista de consulta (sai de /ConsultaNotasFiscaisEmitidas/Details/...)."""
    driver.get(URL_CONSULTA_NOTAS)
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.ID, "Filtro_Cnpj"))
    )
    if "/Details/" in (driver.current_url or ""):
        driver.get(URL_CONSULTA_NOTAS)
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.ID, "Filtro_Cnpj"))
        )


# --- Configurações iniciais ---

dotenv_path = "../data/secure/.env"
load_dotenv(dotenv_path)
EMAIL = os.getenv("LOGIN2")
SENHA = os.getenv("SENHA_ISSE")

# Configurações do navegador Chrome
options = Options()
options.add_argument("--start-maximized")

# Detecta o sistema operacional e configura o driver apropriado
if sys.platform == "win32":
    # Windows: usa o caminho manual (mantém compatibilidade)
    chromedriver_path = r"C:\Users\User\Desktop\Repositorios\Automações\src\others\chromedriver.exe"
    service = Service(chromedriver_path)
    driver = webdriver.Chrome(service=service, options=options)
else:
    # Mac/Linux: usa webdriver-manager (baixa automaticamente)
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

# --- Login ---

driver.get("https://sso.maringa.pr.gov.br/auth/realms/maringa-externo/protocol/openid-connect/auth?response_type=code&client_id=nfse-maringa&redirect_uri=https://maringa.fintel.com.br/Account/OxyOpenId&state=Nfs")

WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, "//*[@id='username']"))).send_keys(EMAIL)
driver.find_element(By.XPATH, "//*[@id='password']").send_keys(SENHA)
driver.find_element(By.XPATH, "//*[@id='password']").send_keys(Keys.RETURN)
time.sleep(5)

# --- Lê os dados da planilha ---
df = pd.read_csv("../data/output/clientes_com_cpf.csv")

# Motivo no <select id="Motivo">: posição 1-based da <option> (//*[@id="Motivo"]/option[N]).
# Padrão 3 = option[3]; altere aqui quando precisar outro motivo.
MOTIVO_OPCAO_INDICE = 3

# Texto em //*[@id="DescricaoMotivo"] — altere quando precisar outro texto.
DESCRICAO_MOTIVO = (
    "Nota fiscal duplicada, serviços prestados em dezembro e lançados em duplicata por engano agora em março"
)

# Telas lentas / SPA: espera longa para abrir detalhe da nota e painel de ações
WAIT_DETALHE_NOTA_S = 45

# DataFrames para guardar resultados
df_sem_cpf = pd.DataFrame(columns=df.columns)
df_notas_removidas = pd.DataFrame(columns=df.columns)

for index, row in df.iterrows():
    time.sleep(1.5)
    ir_para_consulta_notas(driver)

    cpf = row["CPF"]
    if cpf == "000.000.000-00":
        print(f"CPF inválido, pulando: {cpf} (linha {index})")
        df_sem_cpf = pd.concat([df_sem_cpf, pd.DataFrame([row])], ignore_index=True)
        continue

    print(f"Processando exclusão — CPF: {cpf} (linha {index})")

    # Filtro por CPF/CNPJ na consulta (//*[@id="Filtro_Cnpj"])
    filtro_cnpj = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "Filtro_Cnpj"))
    )
    filtro_cnpj.clear()
    filtro_cnpj.send_keys(cpf)

    # Botão pesquisar / aplicar filtro — //*[@id="FormPrincipal"]/div/div[2]/div/button[1]
    btn_filtrar = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="FormPrincipal"]/div/div[2]/div/button[1]')
        )
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", btn_filtrar)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", btn_filtrar)
    time.sleep(1.5)

    # Link na primeira coluna da grid (não é <button>)
    # //*[@id="tabelaConsulta"]/table/tbody/tr/td[1]/div/a[1]
    link_nota = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="tabelaConsulta"]/table/tbody/tr/td[1]/div/a[1]')
        )
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", link_nota)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", link_nota)

    if len(driver.window_handles) > 1:
        driver.switch_to.window(driver.window_handles[-1])

    WebDriverWait(driver, WAIT_DETALHE_NOTA_S).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    WebDriverWait(driver, WAIT_DETALHE_NOTA_S).until(
        EC.presence_of_element_located((By.ID, "acoes_usuario"))
    )
    btn_acao = obter_segundo_controle_acoes_usuario(driver, WAIT_DETALHE_NOTA_S)
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn_acao)
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", btn_acao)
    time.sleep(2)

    # Campo motivo — //*[@id="Motivo"]; opção //*[@id="Motivo"]/option[N] (N = MOTIVO_OPCAO_INDICE)
    select_motivo = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "Motivo"))
    )
    Select(select_motivo).select_by_index(MOTIVO_OPCAO_INDICE - 1)

    # //*[@id="DescricaoMotivo"]
    campo_desc_motivo = WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "DescricaoMotivo"))
    )
    campo_desc_motivo.clear()
    campo_desc_motivo.send_keys(DESCRICAO_MOTIVO)

    # Confirmar envio — //*[@id="conteudo"]/div/div/div/div[2]/form/div/div[4]/button[2]
    btn_enviar = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable(
            (By.XPATH, '//*[@id="conteudo"]/div/div/div/div[2]/form/div/div[4]/button[2]')
        )
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", btn_enviar)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", btn_enviar)

    # Aguarda carregar; a próxima iteração volta ao início (ConsultaNotasFiscaisEmitidas)
    WebDriverWait(driver, 60).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(2)

    df_notas_removidas = pd.concat(
        [df_notas_removidas, pd.DataFrame([row])], ignore_index=True
    )
    print(f"Exclusão enviada — CPF {cpf}. Próximo: recomeça na consulta.")

    # Se a nota tinha aberto em nova aba, fecha e volta à janela principal
    if len(driver.window_handles) > 1:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])

    # Mesma aba às vezes fica em /Details/... — força lista antes da próxima linha
    ir_para_consulta_notas(driver)

os.makedirs("../data/output", exist_ok=True)
df_sem_cpf.to_csv("../data/output/sem_cpf_remove.csv", index=False)
if not df_notas_removidas.empty:
    df_notas_removidas.to_csv("../data/output/notas_removidas.csv", index=False)
print("Encerrado.")
